# Day 9: Schaefer Network Mapping — Which Brain Networks Benefit from Memory?
**Goal:** Map the per-parcel memory benefits from Day 8 onto Schaefer network labels to identify which brain networks benefit most from temporal memory.

**Why this matters:** Our Day 8 results showed 61.4% of parcels benefit from memory (mean ΔR = +0.026). But parcel numbers (842, 117, 529...) don't tell a neuroscience story. Mapping them to networks like Default Mode Network (DMN), frontoparietal, and temporal association areas connects our engineering result to the neuroscience of narrative comprehension and episodic memory.

**What we'll do:**
1. Load Schaefer 1000 atlas with network labels
2. Map Day 8 results to 7 canonical Yeo networks
3. Statistical testing: which networks benefit significantly from memory?
4. Visualize memory benefit on the cortical surface
5. Connect findings to the hippocampal/PFC encoding hypothesis

**Runtime:** CPU is fine for this notebook (analysis only, no training)

In [ ]:
# === INSTALL DEPENDENCIES ===
!pip install nibabel nilearn matplotlib seaborn scipy -q
!pip install scikit-learn statsmodels -q

In [ ]:
# === SETUP ===
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json
from scipy import stats

PROJECT_DIR = '/content/drive/MyDrive/Research/memory-brain-encoding'
DAY8_DIR = os.path.join(PROJECT_DIR, 'day8_results')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'day9_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load Day 8 results
with open(os.path.join(DAY8_DIR, 'day8_results.json')) as f:
    day8 = json.load(f)

print(f"Day 8 recap:")
print(f"  Subject: {day8['data']['subject']}, Task: {day8['data']['task']}")
print(f"  Parcels: {day8['data']['n_parcels']}")
print(f"  Memory benefit: {day8['memory_benefit']['pct_parcels_improved']:.1f}% parcels improved")
print(f"  Mean delta R: {day8['memory_benefit']['mean_delta_R']:+.4f}")
print(f"  Top parcels: {day8['memory_benefit']['top10_parcels']}")

---
## 1. Load Schaefer Atlas Network Labels

The Schaefer 2018 atlas parcellates the cortex into 1000 regions, each assigned to one of 7 Yeo networks:
- **Visual** (Vis)
- **Somatomotor** (SomMot)
- **Dorsal Attention** (DorsAttn)
- **Ventral Attention / Salience** (SalVentAttn)
- **Limbic**
- **Frontoparietal / Control** (Cont)
- **Default Mode Network** (Default)

Our hypothesis: **Default Mode** and **Frontoparietal** networks should benefit most from memory, since they're involved in narrative comprehension, episodic memory, and integrating information over time.

In [ ]:
from nilearn import datasets

# Download Schaefer atlas
schaefer = datasets.fetch_atlas_schaefer_2018(
    n_rois=1000, yeo_networks=7, resolution_mm=2
)

# Parse network labels from parcel names
# Format: "7Networks_LH_Vis_1" -> network = "Vis"
labels = [label.decode() if isinstance(label, bytes) else label 
          for label in schaefer.labels]

# Extract network name for each parcel
networks = []
hemispheres = []
for label in labels:
    parts = label.split('_')
    # Format: 7Networks_LH_NetworkName_RegionNum
    if len(parts) >= 3:
        hemispheres.append(parts[1])  # LH or RH
        networks.append(parts[2])     # Network name
    else:
        hemispheres.append('Unknown')
        networks.append('Unknown')

# Map to full network names
network_fullnames = {
    'Vis': 'Visual',
    'SomMot': 'Somatomotor', 
    'DorsAttn': 'Dorsal Attention',
    'SalVentAttn': 'Salience/Ventral Attention',
    'Limbic': 'Limbic',
    'Cont': 'Frontoparietal Control',
    'Default': 'Default Mode Network',
}

network_colors = {
    'Visual': '#781286',
    'Somatomotor': '#4682B4',
    'Dorsal Attention': '#00760E',
    'Salience/Ventral Attention': '#C43AFA',
    'Limbic': '#DCF8A4',
    'Frontoparietal Control': '#E69422',
    'Default Mode Network': '#CD3E4E',
}

# Count parcels per network
from collections import Counter
network_counts = Counter(networks)
print("Parcels per network:")
for net, count in sorted(network_counts.items(), key=lambda x: -x[1]):
    fullname = network_fullnames.get(net, net)
    print(f"  {fullname}: {count} parcels")

print(f"\nTotal labeled parcels: {len(networks)}")
print(f"Hemispheres: {Counter(hemispheres)}")

---
## 2. Load Per-Parcel Memory Benefits from Day 8

We need the per-parcel correlation values from both models to compute delta R for each parcel.

In [ ]:
import torch

# Load both model histories
mem_ckpt = torch.load(os.path.join(DAY8_DIR, 'day8_model_memory.pt'), 
                       map_location='cpu', weights_only=False)
nomem_ckpt = torch.load(os.path.join(DAY8_DIR, 'day8_model_nomemory.pt'), 
                          map_location='cpu', weights_only=False)

mem_corrs = np.array(mem_ckpt['history']['final_parcel_corrs'])
nomem_corrs = np.array(nomem_ckpt['history']['final_parcel_corrs'])

# Ensure same length
n_parcels = min(len(mem_corrs), len(nomem_corrs), len(networks))
mem_corrs = mem_corrs[:n_parcels]
nomem_corrs = nomem_corrs[:n_parcels]
parcel_networks = networks[:n_parcels]
parcel_hemispheres = hemispheres[:n_parcels]

delta_corrs = mem_corrs - nomem_corrs

print(f"Loaded {n_parcels} parcels")
print(f"Memory model mean R: {mem_corrs.mean():.4f}")
print(f"No-memory model mean R: {nomem_corrs.mean():.4f}")
print(f"Mean delta R: {delta_corrs.mean():.4f}")
print(f"Parcels improved: {(delta_corrs > 0).sum()}/{n_parcels} ({(delta_corrs > 0).mean()*100:.1f}%)")

---
## 3. Network-Level Analysis: Which Networks Benefit from Memory?

This is the core analysis. We group parcels by network and test whether memory provides a significant benefit in each network.

In [ ]:
# Group delta_corrs by network
network_deltas = {}
for i in range(n_parcels):
    net = network_fullnames.get(parcel_networks[i], parcel_networks[i])
    if net not in network_deltas:
        network_deltas[net] = []
    network_deltas[net].append(delta_corrs[i])

# Compute statistics per network
print(f"{'Network':<30} {'N':>4} {'Mean ΔR':>10} {'Median ΔR':>10} {'% Improved':>12} {'t-stat':>8} {'p-value':>10}")
print("=" * 90)

network_stats = {}
for net in sorted(network_deltas.keys()):
    deltas = np.array(network_deltas[net])
    n = len(deltas)
    mean_d = deltas.mean()
    median_d = np.median(deltas)
    pct_improved = (deltas > 0).mean() * 100
    
    # One-sample t-test: is mean delta significantly > 0?
    t_stat, p_val = stats.ttest_1samp(deltas, 0)
    # One-sided p-value (we care about improvement)
    p_val_one = p_val / 2 if t_stat > 0 else 1 - p_val / 2
    
    sig = ''
    if p_val_one < 0.001:
        sig = '***'
    elif p_val_one < 0.01:
        sig = '**'
    elif p_val_one < 0.05:
        sig = '*'
    
    network_stats[net] = {
        'n': n, 'mean': mean_d, 'median': median_d,
        'pct_improved': pct_improved, 't_stat': t_stat, 
        'p_value': p_val_one, 'deltas': deltas
    }
    
    print(f"{net:<30} {n:>4} {mean_d:>+10.4f} {median_d:>+10.4f} {pct_improved:>11.1f}% {t_stat:>8.2f} {p_val_one:>10.4f} {sig}")

# Rank networks by mean delta
print("\n" + "=" * 50)
print("NETWORKS RANKED BY MEMORY BENEFIT:")
print("=" * 50)
ranked = sorted(network_stats.items(), key=lambda x: -x[1]['mean'])
for rank, (net, stat) in enumerate(ranked, 1):
    sig = '***' if stat['p_value'] < 0.001 else '**' if stat['p_value'] < 0.01 else '*' if stat['p_value'] < 0.05 else 'ns'
    print(f"  {rank}. {net}: ΔR = {stat['mean']:+.4f} ({sig})")

---
## 4. Visualization: Network-Level Memory Benefits

Three key plots:
1. Bar chart of mean ΔR per network with significance markers
2. Violin plots showing the distribution of ΔR within each network
3. Hemisphere comparison (left vs right brain)

In [ ]:
fig = plt.figure(figsize=(18, 16))
gs = gridspec.GridSpec(3, 2, hspace=0.4, wspace=0.3)

# 1. Bar chart: Mean delta R per network
ax1 = fig.add_subplot(gs[0, :])
ranked_names = [name for name, _ in ranked]
ranked_means = [stat['mean'] for _, stat in ranked]
ranked_sems = [stat['deltas'].std() / np.sqrt(stat['n']) for _, stat in ranked]
ranked_colors = [network_colors.get(name, '#888888') for name in ranked_names]
ranked_pvals = [stat['p_value'] for _, stat in ranked]

bars = ax1.bar(range(len(ranked_names)), ranked_means, yerr=ranked_sems,
               color=ranked_colors, alpha=0.8, edgecolor='white', linewidth=1.5,
               capsize=5)
ax1.set_xticks(range(len(ranked_names)))
ax1.set_xticklabels(ranked_names, rotation=25, ha='right', fontsize=9)
ax1.set_ylabel('Mean ΔR (Memory − No Memory)')
ax1.set_title('Memory Benefit by Brain Network', fontsize=14, fontweight='bold')
ax1.axhline(y=0, color='black', linewidth=0.5)

# Add significance stars
for i, (name, pval) in enumerate(zip(ranked_names, ranked_pvals)):
    y = ranked_means[i] + ranked_sems[i] + 0.003
    if pval < 0.001:
        ax1.text(i, y, '***', ha='center', fontsize=12, fontweight='bold')
    elif pval < 0.01:
        ax1.text(i, y, '**', ha='center', fontsize=12, fontweight='bold')
    elif pval < 0.05:
        ax1.text(i, y, '*', ha='center', fontsize=12, fontweight='bold')

# 2. Violin plots
ax2 = fig.add_subplot(gs[1, :])
violin_data = [network_stats[name]['deltas'] for name in ranked_names]
parts = ax2.violinplot(violin_data, positions=range(len(ranked_names)),
                        showmeans=True, showmedians=True)

for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(ranked_colors[i])
    pc.set_alpha(0.7)

ax2.set_xticks(range(len(ranked_names)))
ax2.set_xticklabels(ranked_names, rotation=25, ha='right', fontsize=9)
ax2.set_ylabel('ΔR (Memory − No Memory)')
ax2.set_title('Distribution of Memory Benefit Within Each Network', fontsize=12)
ax2.axhline(y=0, color='red', linewidth=0.5, linestyle='--')

# 3. Hemisphere comparison
ax3 = fig.add_subplot(gs[2, 0])
lh_deltas = [delta_corrs[i] for i in range(n_parcels) if parcel_hemispheres[i] == 'LH']
rh_deltas = [delta_corrs[i] for i in range(n_parcels) if parcel_hemispheres[i] == 'RH']

lh_mean = np.mean(lh_deltas)
rh_mean = np.mean(rh_deltas)
lh_sem = np.std(lh_deltas) / np.sqrt(len(lh_deltas))
rh_sem = np.std(rh_deltas) / np.sqrt(len(rh_deltas))

ax3.bar(['Left Hemisphere', 'Right Hemisphere'], [lh_mean, rh_mean],
        yerr=[lh_sem, rh_sem], color=['#2196F3', '#FF5722'], alpha=0.8, capsize=10)
ax3.set_ylabel('Mean ΔR')
ax3.set_title('Memory Benefit by Hemisphere')
ax3.axhline(y=0, color='black', linewidth=0.5)

# Hemisphere t-test
t_hemi, p_hemi = stats.ttest_ind(lh_deltas, rh_deltas)
ax3.text(0.5, 0.95, f'LH vs RH: t={t_hemi:.2f}, p={p_hemi:.3f}',
         transform=ax3.transAxes, ha='center', va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# 4. Memory vs No-Memory scatter per network
ax4 = fig.add_subplot(gs[2, 1])
for net in ranked_names:
    mask = [network_fullnames.get(parcel_networks[i], parcel_networks[i]) == net 
            for i in range(n_parcels)]
    mask = np.array(mask)
    ax4.scatter(nomem_corrs[mask], mem_corrs[mask], 
               color=network_colors.get(net, '#888'), alpha=0.4, s=15, label=net)

# Add diagonal line
lims = [min(ax4.get_xlim()[0], ax4.get_ylim()[0]),
        max(ax4.get_xlim()[1], ax4.get_ylim()[1])]
ax4.plot(lims, lims, 'k--', alpha=0.3, linewidth=1)
ax4.set_xlabel('R (No Memory)')
ax4.set_ylabel('R (Memory)')
ax4.set_title('Per-Parcel: Memory vs No Memory')
ax4.legend(fontsize=6, loc='lower right')

plt.suptitle('Day 9: Brain Network Analysis of Memory Benefit',
             fontsize=15, fontweight='bold', y=1.02)
plt.savefig(os.path.join(RESULTS_DIR, 'day9_network_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Top and Bottom Parcels: Detailed Breakdown

Which specific brain regions benefit most (and least) from memory? This helps us understand what kind of neural computation memory augments.

In [ ]:
# Top 20 parcels benefiting from memory
print("TOP 20 PARCELS (most memory benefit):")
print(f"{'Rank':<5} {'Parcel':<8} {'Network':<30} {'Hemi':<5} {'ΔR':>8} {'R(mem)':>8} {'R(nomem)':>8}")
print("-" * 80)
top_idx = np.argsort(delta_corrs)[::-1][:20]
for rank, idx in enumerate(top_idx, 1):
    net = network_fullnames.get(parcel_networks[idx], parcel_networks[idx])
    hemi = parcel_hemispheres[idx]
    print(f"{rank:<5} {idx:<8} {net:<30} {hemi:<5} {delta_corrs[idx]:>+8.4f} {mem_corrs[idx]:>8.4f} {nomem_corrs[idx]:>8.4f}")

print()
print("BOTTOM 20 PARCELS (memory hurts most):")
print(f"{'Rank':<5} {'Parcel':<8} {'Network':<30} {'Hemi':<5} {'ΔR':>8} {'R(mem)':>8} {'R(nomem)':>8}")
print("-" * 80)
bottom_idx = np.argsort(delta_corrs)[:20]
for rank, idx in enumerate(bottom_idx, 1):
    net = network_fullnames.get(parcel_networks[idx], parcel_networks[idx])
    hemi = parcel_hemispheres[idx]
    print(f"{rank:<5} {idx:<8} {net:<30} {hemi:<5} {delta_corrs[idx]:>+8.4f} {mem_corrs[idx]:>8.4f} {nomem_corrs[idx]:>8.4f}")

# Network composition of top vs bottom parcels
print("\n" + "=" * 50)
print("NETWORK COMPOSITION:")
print("=" * 50)
top50_nets = Counter([network_fullnames.get(parcel_networks[i], parcel_networks[i]) 
                       for i in np.argsort(delta_corrs)[-50:]])
bot50_nets = Counter([network_fullnames.get(parcel_networks[i], parcel_networks[i]) 
                       for i in np.argsort(delta_corrs)[:50]])

print(f"\n{'Network':<30} {'In Top 50':>10} {'In Bottom 50':>12} {'Ratio':>8}")
print("-" * 65)
all_nets = set(list(top50_nets.keys()) + list(bot50_nets.keys()))
for net in sorted(all_nets):
    top_n = top50_nets.get(net, 0)
    bot_n = bot50_nets.get(net, 0)
    ratio = top_n / max(bot_n, 1)
    marker = ' <<<' if ratio > 2 else ' !!!' if ratio < 0.5 else ''
    print(f"{net:<30} {top_n:>10} {bot_n:>12} {ratio:>8.2f}{marker}")

---
## 6. Cortical Surface Visualization

Map the memory benefit (ΔR) onto the brain surface to see the spatial pattern.

In [ ]:
from nilearn import plotting, surface, datasets as nl_datasets

# Create a parcel-level map of delta R
# Map each parcel's delta R back to the atlas volume
atlas_img = schaefer.maps
atlas_data = atlas_img.get_fdata()

# Create a new image where each parcel's value is its delta R
benefit_data = np.zeros_like(atlas_data, dtype=np.float32)
for i in range(n_parcels):
    benefit_data[atlas_data == (i + 1)] = delta_corrs[i]

import nibabel as nib
benefit_img = nib.Nifti1Image(benefit_data, atlas_img.affine, atlas_img.header)

# Plot on cortical surface
fig, axes = plt.subplots(2, 2, figsize=(16, 12),
                          subplot_kw={'projection': '3d'})

# Get fsaverage for surface plotting
fsaverage = nl_datasets.fetch_surf_fsaverage()

for idx, (hemi, view) in enumerate([('left', 'lateral'), ('left', 'medial'),
                                      ('right', 'lateral'), ('right', 'medial')]):
    ax = axes[idx // 2, idx % 2]
    
    # Project volume to surface
    surf_mesh = fsaverage[f'pial_{hemi}']
    texture = surface.vol_to_surf(benefit_img, surf_mesh)
    
    plotting.plot_surf_stat_map(
        surf_mesh, texture,
        hemi=hemi, view=view,
        cmap='RdBu_r', colorbar=True,
        vmax=np.percentile(np.abs(delta_corrs), 95),
        title=f'{hemi.upper()} - {view}',
        figure=fig, axes=ax,
        threshold=0.01
    )

plt.suptitle('Memory Benefit (ΔR) on Cortical Surface\nRed = memory helps, Blue = memory hurts',
             fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(RESULTS_DIR, 'day9_cortical_surface.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Saved cortical surface visualization")

---
## 7. Statistical Summary and Paper-Ready Results

In [ ]:
# Comprehensive statistical summary
print("=" * 70)
print("DAY 9: STATISTICAL SUMMARY")
print("=" * 70)

# Overall effect
t_overall, p_overall = stats.ttest_1samp(delta_corrs, 0)
print(f"\nOverall memory benefit:")
print(f"  Mean ΔR = {delta_corrs.mean():+.4f} (SE = {delta_corrs.std()/np.sqrt(len(delta_corrs)):.4f})")
print(f"  t({len(delta_corrs)-1}) = {t_overall:.3f}, p = {p_overall/2:.6f} (one-sided)")
print(f"  Cohen's d = {delta_corrs.mean() / delta_corrs.std():.3f}")
print(f"  {(delta_corrs > 0).sum()}/{len(delta_corrs)} parcels improved ({(delta_corrs>0).mean()*100:.1f}%)")

# ANOVA: does network identity predict memory benefit?
from scipy.stats import f_oneway
network_groups = [network_stats[net]['deltas'] for net in network_stats]
f_stat, p_anova = f_oneway(*network_groups)
print(f"\nNetwork effect (one-way ANOVA):")
print(f"  F({len(network_stats)-1}, {n_parcels-len(network_stats)}) = {f_stat:.3f}, p = {p_anova:.6f}")
if p_anova < 0.05:
    print(f"  -> Networks differ significantly in memory benefit!")

# Effect sizes per network
print(f"\nEffect sizes (Cohen's d) per network:")
for net, stat in sorted(network_stats.items(), key=lambda x: -x[1]['mean']):
    d = stat['mean'] / stat['deltas'].std() if stat['deltas'].std() > 0 else 0
    size = 'large' if abs(d) > 0.8 else 'medium' if abs(d) > 0.5 else 'small'
    print(f"  {net:<30} d = {d:+.3f} ({size})")

---
## 8. Save Results

In [ ]:
# Save results
results = {
    'day': 9,
    'date': __import__('time').strftime('%Y-%m-%d'),
    'task': 'Network-level analysis of memory benefit',
    'overall': {
        'mean_delta_R': float(delta_corrs.mean()),
        'se_delta_R': float(delta_corrs.std() / np.sqrt(len(delta_corrs))),
        't_stat': float(t_overall),
        'p_value_onesided': float(p_overall / 2),
        'cohens_d': float(delta_corrs.mean() / delta_corrs.std()),
        'pct_improved': float((delta_corrs > 0).mean() * 100),
    },
    'anova': {
        'F_stat': float(f_stat),
        'p_value': float(p_anova),
    },
    'networks': {},
    'hemisphere': {
        'LH_mean': float(np.mean(lh_deltas)),
        'RH_mean': float(np.mean(rh_deltas)),
        't_stat': float(t_hemi),
        'p_value': float(p_hemi),
    },
    'top10_parcels_with_labels': [],
}

for net, stat in network_stats.items():
    d = stat['mean'] / stat['deltas'].std() if stat['deltas'].std() > 0 else 0
    results['networks'][net] = {
        'n_parcels': int(stat['n']),
        'mean_delta_R': float(stat['mean']),
        'median_delta_R': float(stat['median']),
        'pct_improved': float(stat['pct_improved']),
        't_stat': float(stat['t_stat']),
        'p_value': float(stat['p_value']),
        'cohens_d': float(d),
    }

for idx in np.argsort(delta_corrs)[-10:][::-1]:
    results['top10_parcels_with_labels'].append({
        'parcel': int(idx),
        'network': network_fullnames.get(parcel_networks[idx], parcel_networks[idx]),
        'hemisphere': parcel_hemispheres[idx],
        'delta_R': float(delta_corrs[idx]),
    })

with open(os.path.join(RESULTS_DIR, 'day9_results.json'), 'w') as f:
    json.dump(results, f, indent=2)

# Save delta corrs with labels for future use
np.savez(os.path.join(RESULTS_DIR, 'day9_parcel_data.npz'),
         delta_corrs=delta_corrs,
         mem_corrs=mem_corrs,
         nomem_corrs=nomem_corrs,
         networks=np.array(parcel_networks),
         hemispheres=np.array(parcel_hemispheres))

print("Saved to Google Drive:")
print(f"  {RESULTS_DIR}/")
print(f"  ├── day9_network_analysis.png")
print(f"  ├── day9_cortical_surface.png")
print(f"  ├── day9_results.json")
print(f"  └── day9_parcel_data.npz")
print()
print("=" * 60)
print("DAY 9 COMPLETE")
print("=" * 60)

# Print the key finding
ranked = sorted(network_stats.items(), key=lambda x: -x[1]['mean'])
print(f"\nKey finding: Top networks benefiting from memory:")
for i, (net, stat) in enumerate(ranked[:3], 1):
    sig = '***' if stat['p_value'] < 0.001 else '**' if stat['p_value'] < 0.01 else '*' if stat['p_value'] < 0.05 else 'ns'
    print(f"  {i}. {net}: ΔR = {stat['mean']:+.4f} (p = {stat['p_value']:.4f}) {sig}")

print(f"\nReady for Day 10: Real TRIBE v2 feature extraction from Friends video!")